In [ ]:
# 1. INSTALACIÓN 
import os
import sys

# Instalamos SciSpacy y el modelo
!pip install scispacy==0.5.4 --no-deps
!pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_sm-0.5.4.tar.gz --no-deps
!pip install pysbd conllu nmslib-metabrainz==2.1.3 --no-deps

# 2. IMPORTS DIRECTOS
import numpy as np
import pandas as pd
import spacy
import scispacy
import en_core_sci_sm

print(f"Versión NumPy: {np.__version__}")
print(f"Versión Pandas: {pd.__version__}")

# 3. CARGA DEL MODELO
nlp = en_core_sci_sm.load()
print("¡Modelo cargado exitosamente!")

In [ ]:
#### LECTURAS DE DATOS

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# Cargamos el dataset principal
train = pd.read_csv('/kaggle/input/competitions/triagegeist/train.csv')
hist = pd.read_csv('/kaggle/input/competitions/triagegeist/patient_history.csv')
d_texto = pd.read_csv('/kaggle/input/competitions/triagegeist/chief_complaints.csv')

test = pd.read_csv('/kaggle/input/competitions/triagegeist/test.csv')

# Vemos qué columnas tenemos para analizar
print("Columnas del dataset:")
print(train.columns.tolist())



In [ ]:
######## DATOS DE TEXTO - VAMOS A HACER UNA EXTRACCION DE SINTOMAS CON SciSpacy
#####UN MODELO MEDICO ESPECIFICO 


# 1. Cargamos el modelo
nlp = spacy.load("en_core_sci_sm")

# 2. Lista de "Stop Words" médicas (palabras que SciSpacy detecta pero no son síntomas)
# Agregamos 'patient' y 'denies' para que no ensucien el modelo estadístico
import unicodedata
import pandas as pd

# 1. Definimos el ruido ampliado (metadatos y tiempos)
RUIDO_MEDICO = {
    "patient", "denies", "history", "hospital", "admission", 
    "year", "old", "male", "female", "arrival", "onset", 
    "today", "day", "hour", "constant", "intermittent", 
    "moderate", "mild", "severe", "acute", "chronic",
    "paperwork", "advice", "plan"
}

# 2. Función para limpiar caracteres extraños
def limpiar_codificacion(texto):
    if pd.isna(texto): return ""
    # Elimina acentos y caracteres raros (como ï¼Œ)
    texto = unicodedata.normalize('NFKD', str(texto))
    texto = texto.encode('ascii', 'ignore').decode('ascii')
    return texto.lower().strip()

# 3. Función de extracción usando LEMMAS --- para que palabras similares me las agrupe igual
def extraer_con_lemmas(texto):
    texto_limpio = limpiar_codificacion(texto)
    if not texto_limpio: return []
    
    doc = nlp(texto_limpio)
    sintomas = []
    
    for ent in doc.ents:
        # Usamos .lemma_ para normalizar a la raíz
        lemma = ent.lemma_.strip()
        
        # Filtramos si el lemma está en ruido o es muy corto
        if lemma not in RUIDO_MEDICO and len(lemma) > 2:
            sintomas.append(lemma)
            
    return list(set(sintomas)) # Eliminamos duplicados dentro de la misma frase

# Tomamos una muestra aleatoria para validar
#muestra = d_texto.sample(n=1000, random_state=42).copy()
# 4. Aplicamos a la muestra de 1000
#muestra['sintomas_lemmas'] = muestra['chief_complaint_raw'].apply(extraer_con_lemmas)

# Guardamos la nueva versión para revisarla
#muestra.to_csv('muestra_triage_lemmas.csv', index=False)

# Vemos los resultados para ver si SciSpacy está captando lo que queremos
#print(muestra.head(20))

# 3. Procesamos archivo de quejas (chief_complaints)
#d_texto['lista_sintomas'] = d_texto['chief_complaint_raw'].apply(extraer_entidades_puras)
#print("Ejemplo de extracción:")
#print(d_texto[['chief_complaint_raw', 'lista_sintomas']].head())

#AHORA LO HACEMOS PARA TODOS LOS DATOS, NO SOLO PARA LA MEUSTRA DE 100000 CASOS


d_texto['sintomas_lemmas'] = d_texto['chief_complaint_raw'].apply(extraer_con_lemmas)

# Guardamos la nueva versión para revisarla
d_texto.to_csv('triage_lemmas_todosDatos.csv', index=False)


In [ ]:
# ############## LO QUE OBTUVE ANTES, LO ANALICE POR CSV QUITANDO REPETIDOS Y BUSCANDO PALABRAS RUIDO Y SEMEJANTES AGRUPANDO EN CATEGORIAS --- LIMPIANDO Y CATEGORIZANDO
# ##### APUNTES

# ## 1 - DERMATOLOGICO

# #RUIDO_MEDICO = {"worsen","worsening","movement","spread","localised","localise","sign","with","associated","minor","small","acute","request","removal","check"}

# # Diccionario de Mapeo Clínico
# #mapeo_derm = {
#     'anaphylaxis': 'ANAPHYLAXIS_URTICARIA',
#     'urticaria': 'ANAPHYLAXIS_URTICARIA',
#     'stevens-johnson syndrome': 'CRITICAL_SKIN_FAILURE',
#     'toxic epidermal': 'CRITICAL_SKIN_FAILURE',
#     'bullous pemphigoid': 'CRITICAL_SKIN_FAILURE',
#     'necrotise fasciitis': 'SEVERE_INFECTION',
#     'sepsis': 'SEVERE_INFECTION',
#     'meningococcal': 'SEVERE_INFECTION',
#     'cellulitis': 'CELLULITIS_INFECTION',
#     'wound infection': 'CELLULITIS_INFECTION',
#     'infected': 'CELLULITIS_INFECTION',
#     'tick bite': 'INFESTATION_BITE',
#     'erythema migran': 'INFESTATION_BITE',
#     'shingle': 'INFESTATION_BITE',
#     'eczema': 'DERMATITIS_RASH',
#     'rash': 'DERMATITIS_RASH',
#     'dermatitis': 'DERMATITIS_RASH',
#     'sunburn': 'MINOR_CONDITIONS',
#     'ingrown': 'MINOR_CONDITIONS'
# }

# ## 2 - CARDIOLOGIA

# #RUIDO_MEDICO = {"cardiology", "monitoring", "inquiry", "follow-up", "stable","constant", "hours", "days", "minor", "mild", "suspected", "worsening","antihypertensive"

# #mapeo_cardio = {
#     'nstemi': 'ACUTE_CORONARY',
#     'chest pain': 'ACUTE_CORONARY',
#     'exertional': 'ACUTE_CORONARY',
#     'angina': 'ACUTE_CORONARY',
#     'dyspnoea': 'RESPIRATORY_CARDIAC',
#     'syncope': 'HEMODYNAMIC_INSTABILITY',
#     'decompensated': 'HEMODYNAMIC_INSTABILITY',
#     'palpitations': 'HEMODYNAMIC_INSTABILITY',
#     'atrial fibrillation': 'HEMODYNAMIC_INSTABILITY',
#     'blood pressure': 'HYPERTENSIVE_EVENT',
#     'diaphoresis': 'HEMODYNAMIC_INSTABILITY' # Sudoración fría es clave en cardio
# #}

# ##3 - ENDOCRINOLOGIA

# #RUIDO_MEDICO = {"symptomatic", "features", "management", "query", "resolved",""constant", "hours", "moderate", "mild","thyroid-related","related"}

# #mapeo_endo = {
#     'hypoglycaemia': 'GLYCEMIC_CRISIS',
#     'hyperglycaemia': 'GLYCEMIC_CRISIS',
#     'glucose': 'GLYCEMIC_CRISIS',
#     'diabetic': 'GLYCEMIC_CRISIS',
#     'coma': 'ENDOCRINE_EMERGENCY',
#     'thyroid storm': 'ENDOCRINE_EMERGENCY',
#     'adrenal': 'ENDOCRINE_EMERGENCY',
#     'cushing': 'METABOLIC_SYNDROME',
#     'weight': 'METABOLIC_SYNDROME',
#     'thyroid': 'THYROID_DISORDER'
# #}


# ##4 - ENT

# #NUEVO_RUIDO_ENT = {
#     "moderate", "constant", "worsening", "hours", "movement", 
#     "temporal", "ear", "nose", "throat", "mild", "severe", 
#     "blocked", "irritation", "check", "removal", "wax"
# #}

# RUIDO_MEDICO.update(NUEVO_RUIDO_ENT)

# mapeo_ent = {
#     # EMERGENCIAS DE VÍA AÉREA (Prioridad 1)
#     'epiglottitis': 'AIRWAY_EMERGENCY',
#     'airway': 'AIRWAY_EMERGENCY',
#     'trismus': 'AIRWAY_EMERGENCY',
    
#     # INFECCIONES AGUDAS (Prioridad 2-3)
#     'mastoiditis': 'ACUTE_INFECTION_ENT',
#     'tonsillitis': 'ACUTE_INFECTION_ENT',
#     'sinusitis': 'ACUTE_INFECTION_ENT',
    
#     # CUERPOS EXTRAÑOS
#     'foreign body': 'FOREIGN_BODY_ENT',
    
#     # HEMORRAGIAS
#     'epistaxis': 'NASAL_HEMORRHAGE',
    
#     # SÍNTOMAS COMUNES / BAJA URGENCIA
#     'ear pain': 'EAR_DISCOMFORT',
#     'hearing': 'EAR_DISCOMFORT',
#     'runny nose': 'NASAL_SYMPTOMS',
#     'nasal congestion': 'NASAL_SYMPTOMS',
#     'sore throat': 'THROAT_SYMPTOMS'
# }


# ## 5 -  GASTROENTEROLOGO

# NUEVO_RUIDO_GASTRO = {
#     "moderate", "constant", "mild", "hours", "days", 
#     "clinically", "follow-up", "movement", "new-onset", 
#     "non-bloody", "organ", "abdominal" # Abdominal es redundante en Gastro
# }

# RUIDO_MEDICO.update(NUEVO_RUIDO_GASTRO)

# mapeo_gastro = {
#     # EMERGENCIAS HEMORRÁGICAS (Prioridad 1)
#     'haematemesis': 'GI_BLEED_UPPER',
#     'coffee-ground': 'GI_BLEED_UPPER',
#     'bleed': 'GI_BLEED_GENERAL',
    
#     # EMERGENCIAS DE ÓRGANO / AGUDAS
#     'acute abdomen': 'ACUTE_ABDOMEN_SURGICAL',
#     'obstruction': 'BOWEL_OBSTRUCTION',
#     'hepatic failure': 'HEPATIC_CRISIS',
#     'pancreatitis': 'ACUTE_PANCREATITIS',
#     'ischaemia': 'MESENTERIC_ISCHAEMIA',
    
#     # SÍNTOMAS AGUDOS / INFECCIOSOS
#     'jaundice': 'BILIARY_LIVER_SYMPTOMS',
#     'diarrhoe': 'DIARRHEA_GASTROENTERITIS',
#     'rigors': 'SYSTEMIC_INFECTION_GASTRO',
    
#     # SÍNTOMAS FUNCIONALES / BAJA URGENCIA
#     'dyspepsia': 'FUNCTIONAL_GASTRO',
#     'constipation': 'FUNCTIONAL_GASTRO',
#     'bloating': 'FUNCTIONAL_GASTRO',
#     'nausea': 'NAUSEA_VOMITING_GEN',
#     'vomiting': 'NAUSEA_VOMITING_GEN'
# }

# ## 6 - GENITOURINARIO

# NUEVO_RUIDO_GU = {
#     "mild", "uncomplicated", "hours", "movement", "constant",
#     "screening", "prescription", "contraception", "significant",
#     "systemic", "feature", "acute" # Acute es muy común aquí, mejor confiar en el síntoma raíz
# }

# RUIDO_MEDICO.update(NUEVO_RUIDO_GU)

# mapeo_genitourinario = {
#     # EMERGENCIAS CRÍTICAS (Prioridad 1)
#     'urosepsis': 'UROSEPSIS_CRITICAL',
#     'ovarian torsion': 'GONADAL_TORSION_EMERGENCY',
#     'testicular': 'GONADAL_TORSION_EMERGENCY',
#     'scrotal pain': 'GONADAL_TORSION_EMERGENCY',
#     'ischaemia': 'GU_ISCHAEMIA',

#     # CUADROS AGUDOS GRAVES
#     'pyelonephritis': 'ACUTE_PYELONEPHRITIS',
#     'renal colic': 'NEPHROLITHIASIS_COLIC',
    
#     # SÍNTOMAS URINARIOS / INFECCIONES
#     'haematuria': 'URINARY_BLEEDING',
#     'dysuria': 'UTI_SYMPTOMS',
#     'urinary retention': 'ACUTE_URINARY_RETENTION',
    
#     # GINECOLOGÍA / OTROS
#     'vaginal bleed': 'VAGINAL_BLEEDING',
#     'genital discharge': 'GENITAL_INFECTION_STD',
    
#     # SÍNTOMAS SISTÉMICOS ASOCIADOS
#     'fever': 'FEVER_GU_ASSOCIATED',
#     'vomiting': 'NAUSEA_VOMITING_GU'
# }

# ## 7 - INFECCIOSO

# NUEVO_RUIDO_INF = {
#     "mild", "moderate", "constant", "intermittent", "worsening",
#     "hours", "days", "travel", "prophylaxis", "post-exposure",
#     "uncomplicated", "systemic", "feature", "low-risk", "viral"
# }

# RUIDO_MEDICO.update(NUEVO_RUIDO_INF)

# mapeo_infeccioso = {
#     # EMERGENCIAS DE VIDA (Prioridad 1)
#     'meningitis': 'CENTRAL_NERVOUS_SYSTEM_INF',
#     'mental status': 'CENTRAL_NERVOUS_SYSTEM_INF',
#     'necrotising fasciitis': 'NECROTISING_SOFT_TISSUE_INF',
    
#     # CUADROS AGUDOS ESPECÍFICOS
#     'dengue': 'TROPICAL_INF_DENGUE',
#     'hepatitis': 'HEPATIC_INF',
    
#     # INFECCIONES DE HERIDAS / LOCALES
#     'wound infection': 'BACTERIAL_WOUND_INF',
#     'abscess': 'LOCALIZED_BACTERIAL_INF',
    
#     # SÍNTOMAS GENERALES (Frecuentes en Triage)
#     'fever': 'FEVER_SYNDROME',
#     'rigors': 'FEVER_SYNDROME',
#     'diaphoresis': 'FEVER_SYNDROME',
    
#     # CUADROS LEVES
#     'gastro': 'GASTROENTERITIS_INF',
#     'skin rash': 'EXANTHEMA_RASH_INF'
# }

# ## 8 - MUSCULOESQUELETO

# NUEVO_RUIDO_MSK = {
#     "mild", "moderate", "constant", "hours", "days", 
#     "activity", "movement", "weight bearing", "chronic flare", 
#     "physiotherapy", "generalised", "distal", "lateral"
# }

# RUIDO_MEDICO.update(NUEVO_RUIDO_MSK)

# mapeo_msk = {
#     # EMERGENCIAS NEUROLÓGICAS/VASCULARES (Prioridad 1)
#     'spinal cord': 'SPINAL_CORD_CRITICAL',
#     'vascular compromise': 'VASCULAR_EMERGENCY_MSK',
    
#     # INFECCIONES ÓSEAS/ARTICULARES (Grave)
#     'septic arthritis': 'SEPTIC_ARTHRITIS_EMERGENCY',
    
#     # FRACTURAS Y TRAUMA MAYOR
#     'fracture': 'BONE_FRACTURE',
#     'distal radius': 'BONE_FRACTURE', # Muy común en caídas
#     'pelvic': 'MAJOR_TRAUMA_PELVIC',
#     'traumatic': 'MAJOR_TRAUMA_GENERAL',
    
#     # LESIONES AGUDAS COMUNES
#     'ankle': 'JOINT_INJURY_LOWER',
#     'strain': 'MUSCLE_STRAIN_SPRAIN',
#     'bruise': 'CONTUSION_BRUISE',
#     'fall': 'MECHANISM_OF_INJURY_FALL',
    
#     # DOLOR AXIAL / CRÓNICO
#     'back pain': 'AXIAL_SKELETAL_PAIN',
#     'ache': 'NON_SPECIFIC_PAIN_MSK'
# }

# ## 9 - NEUTOLOGICO

# NUEVO_RUIDO_NEURO = {
#     "mild", "moderate", "constant", "hours", "days", 
#     "resolved", "elderly", "patient", "positional", 
#     "signs", "with", "acute", "residual", "memory"
# }

# RUIDO_MEDICO.update(NUEVO_RUIDO_NEURO)

# mapeo_neurologico = {
#     # CÓDIGO ROJO / EMERGENCIAS VITALES
#     'status epilepticus': 'CRITICAL_SEIZURE_STATUS',
#     'thunderclap': 'ACUTE_CEREBROVASCULAR_HEMORRHAGE', # Dolor de cabeza súbito/estallido
#     'coma': 'CONSCIOUSNESS_LOSS_COMA',
#     'locked-in syndrome': 'BRAINSTEM_CRITICAL_SYNDROME',
    
#     # POSIBLE ACV / FOCALIDAD (Prioridad 1-2)
#     'facial droop': 'STROKE_SYMPTOMS_FOCAL',
#     'hand paraesthesia': 'STROKE_SYMPTOMS_FOCAL',
#     'transient': 'TIA_TRANSIENT_ISCHAEMIA',
    
#     # CONVULSIONES Y EPILEPSIA
#     'seizure': 'EPILEPTIC_EVENT',
    
#     # CEFALEAS
#     'migraine': 'SEVERE_HEADACHE_MIGRAINE',
    
#     # MAREOS / VÉRTIGO (Diferenciar central de periférico es clave)
#     'vertigo': 'VESTIBULAR_VERTIGO_SYMPTOMS',
#     'dizziness': 'NON_SPECIFIC_DIZZINESS',
    
#     # INFECCIÓN CNS
#     'meningitis': 'CNS_INFECTION_NEURO',
    
#     # OTROS
#     'confusion': 'ALTERED_MENTAL_STATE',
#     'fever': 'FEVER_NEURO_ASSOCIATED'
# }


# ## 10 - OFTALMOLOGICO

#  NUEVO_RUIDO_OPH = {
#     "mild", "moderate", "constant", "hours", "worsening",
#     "sudden", "bilateral", "eyes", "contact lens", "prescription",
#     "question", "movement"
# }

# RUIDO_MEDICO.update(NUEVO_RUIDO_OPH)

# mapeo_oftalmologico = {
#     # EMERGENCIAS VASCULARES/AGUDAS (Prioridad 1)
#     'sudden loss': 'SUDDEN_VISION_LOSS_EMERGENCY',
#     'central retinal': 'RETINAL_VASCULAR_EMERGENCY',
#     'acute angle': 'ACUTE_GLAUCOMA_EMERGENCY',
    
#     # INFECCIONES GRAVES
#     'orbital cellulitis': 'ORBITAL_CELLULITIS_SEVERE',
    
#     # TRAUMA / CUERPOS EXTRAÑOS
#     'foreign body': 'OCULAR_FOREIGN_BODY',
#     'corneal abrasion': 'CORNEAL_TRAUMA',
#     'corneal': 'CORNEAL_TRAUMA',
    
#     # INFLAMACIÓN / INFECCIÓN SUPERFICIAL
#     'conjunctivitis': 'SUPERFICIAL_EYE_INFECTION',
#     'eyelid swell': 'EYELID_INFLAMMATION',
#     'red painful': 'ACUTE_RED_EYE_SYNDROME',
    
#     # OTROS SÍNTOMAS
#     'photophobia': 'PHOTOPHOBIA_SYMPTOM',
#     'vision': 'VISUAL_DISTURBANCE_GENERIC',
#     'fever': 'FEVER_OPH_ASSOCIATED'
# }

# ## 11- PSIQUIATRICOS
# # Agrega esto a tu set global de RUIDO_MEDICO
# NUEVO_RUIDO_PSIQ = {
#     "mild", "moderate", "constant", "intermittent", "hours", 
#     "days", "chronic", "mental health", "counselling", 
#     "acute", "intent"
# }

# RUIDO_MEDICO.update(NUEVO_RUIDO_PSIQ)

# mapeo_psiquiatrico = {
#     # EMERGENCIAS VITALES
#     'suicidal ideation': 'SELF_HARM_SUICIDAL',
#     'self-harm': 'SELF_HARM_SUICIDAL',
#     'plan': 'SELF_HARM_SUICIDAL',
#     'overdose': 'SUBSTANCE_EMERGENCY',
#     'intoxication': 'SUBSTANCE_EMERGENCY',
#     'alcohol withdrawal': 'SUBSTANCE_EMERGENCY',
    
#     # CUADROS AGUDOS GRAVES
#     'psychosis': 'ACUTE_PSYCHOSIS',
#     'mania': 'ACUTE_PSYCHOSIS',
    
#     # ANSIEDAD Y ESTRÉS
#     'panic attack': 'ANXIETY_PANIC',
#     'anxiety': 'ANXIETY_PANIC',
#     'stress': 'ANXIETY_PANIC',
    
#     # OTROS
#     'insomnia': 'SLEEP_DISORDERS',
#     'depressive': 'DEPRESSIVE_SYMPTOMS'
# }


# ## 12 - RESPIRATORIO

# # Agrega esto a tu set global de RUIDO_MEDICO
# NUEVO_RUIDO_RESP = {
#     "mild", "moderate", "constant", "hours", "days", 
#     "week", "chronic", "movement", "symptoms", "upper",
#     "moderate-to-severe", "rest"
# }

# RUIDO_MEDICO.update(NUEVO_RUIDO_RESP)

# mapeo_respiratorio = {
#     # EMERGENCIAS VITALES
#     'hypoxia': 'RESPIRATORY_FAILURE',
#     'tension': 'RESPIRATORY_FAILURE',
#     'acute severe': 'RESPIRATORY_FAILURE',
    
#     # DIFICULTAD RESPIRATORIA
#     'dyspnoea': 'ACUTE_DYSPNOEA',
#     'haemoptysis': 'AIRWAY_BLEEDING',
    
#     # DOLOR Y OTROS
#     'pleuritic': 'CHEST_PAIN_RESP',
#     'cough': 'CHRONIC_AIRWAY',
#     'night sweats': 'INFECTIOUS_RESP',
#     'inhaler': 'CHRONIC_AIRWAY',
#     'nasal congestion': 'NASAL_SYMPTOMS'
# }

# ## 13 - TRAUMATOLOGIA
# # Agrega esto a tu set global de RUIDO_MEDICO
# NUEVO_RUIDO_TRAUMA = {
#     "ground-level", "high-speed", "mechanism", "hours", "days", 
#     "constant", "intermittent", "small", "stable", "vital",
#     "request", "removal", "deep", "superficial"
# }

# RUIDO_MEDICO.update(NUEVO_RUIDO_TRAUMA)

# mapeo_traumatologia = {
#     # EMERGENCIAS DE ALTA PRIORIDAD
#     'multi-system': 'CRITICAL_TRAUMA',
#     'unresponsive': 'CRITICAL_TRAUMA',
#     'stab': 'PENETRATING_WOUND',
    
#     # TRAUMA CRANEOENCEFÁLICO
#     'head injury': 'HEAD_NEURO_TRAUMA',
#     'head bump': 'HEAD_NEURO_TRAUMA',
#     'facial trauma': 'HEAD_NEURO_TRAUMA',
    
#     # LESIONES ESPECÍFICAS
#     'rib fracture': 'BONE_FRACTURE_TRAUMA',
#     'deep dog': 'PENETRATING_WOUND',
#     'laceration': 'OPEN_WOUND_LACERATION',
    
#     # LESIONES MENORES
#     'splinter': 'MINOR_FOREIGN_BODY',
#     'abrasion': 'SOFT_TISSUE_INJURY',
#     'insect sting': 'MINOR_TRAUMA_REACTION'
# }


# ## 14 - OTROS

# # Agrega esto a tu set global de RUIDO_MEDICO
# NUEVO_RUIDO_OTROS = {
#     "non-urgent", "request", "check", "general", "inquiry",
#     "social", "medical condition", "status", "hours", "days",
#     "moderate", "constant", "worsening", "intermittent", "abnormal"
# }

# RUIDO_MEDICO.update(NUEVO_RUIDO_OTROS)

# mapeo_otros = {
#     # EMERGENCIAS METABÓLICAS
#     'hypothermia': 'METABOLIC_CRITICAL',
#     'dehydration': 'METABOLIC_CRITICAL',
#     'electrolyte': 'METABOLIC_CRITICAL',
    
#     # ESTADOS GENERALES
#     'weight loss': 'SYSTEMIC_GENERAL_SYMPTOMS',
#     'fatigue': 'SYSTEMIC_GENERAL_SYMPTOMS',
#     'fever': 'SYSTEMIC_GENERAL_SYMPTOMS',
    
#     # TOXICOLOGÍA / SUSTANCIAS
#     'substance': 'SUBSTANCE_RELATED_GENERAL',
    
#     # ADMINISTRATIVO / TRÁMITES
#     'non-urgent': 'ADMINISTRATIVE_NON_CLINICAL',
#     'request': 'ADMINISTRATIVE_NON_CLINICAL',
#     'check': 'ADMINISTRATIVE_NON_CLINICAL',
#     'inquiry': 'ADMINISTRATIVE_NON_CLINICAL',
    
#     # MECANISMO DE LESIÓN
#     'fall': 'MECHANISM_OF_INJURY_FALL',
#     'injury': 'MECHANISM_OF_INJURY_FALL'
# }


In [ ]:
####### ESA ANTERIOR ERA EL LISTADO (NO PARA CORRERLO SINO COMO APUNTE) - AHORA SI CORREO Y LIMPIO Y LO CORRO ESTA VEZ SOBRE LOS 80000 CASOS
# 1. CONSOLIDACIÓN DE RUIDO GLOBAL
RUIDO_MEDICO_TOTAL = {
    # Temporalidad y Frecuencia
    'hours', 'days', 'week', 'constant', 'intermittent', 'acute', 'chronic', 
    'worsening', 'stable', 'onset', 'resolved', 'mild', 'moderate', 'severe',
    'moderate-to-severe', 'intermitt', 'temporal',
    # Administrativos y Logística
    'request', 'check', 'inquiry', 'non-urgent', 'prescription', 'screening', 
    'follow-up', 'management', 'query', 'counselling', 'transfusion',
    'medical condition', 'status', 'question', 'removal', 'wax',
    # Contexto, Movimiento y Anatomía (Redundante)
    'movement', 'activity', 'positional', 'weight bearing', 'rest', 'distance', 
    'ground-level', 'high-speed', 'mechanism', 'travel', 'prophylaxis', 
    'post-exposure', 'low-risk', 'uncomplicated', 'blocked', 'irritation',
    'ear', 'nose', 'throat', 'nasal', 'aural', 'oral', 'abdominal', 'organ',
    # Relleno Semántico
    'symptoms', 'features', 'clinical', 'signs with', 'patient', 'elderly', 
    'systemic', 'significant', 'general', 'generalised', 'with', 'clinically',
    'new-onset', 'non-bloody', 'intent', 'feature', 'viral'
}

# 2. MAPEO MAESTRO DE LAS 14 ESPECIALIDADES
MAPEO_MAESTRO_COMPLETO = {
    # 1. DERMATO
    'anaphylaxis': 'ANAPHYLAXIS_URTICARIA', 'urticaria': 'ANAPHYLAXIS_URTICARIA',
    'stevens-johnson': 'CRITICAL_SKIN_FAILURE', 'toxic epidermal': 'CRITICAL_SKIN_FAILURE',
    'bullous pemphigoid': 'CRITICAL_SKIN_FAILURE', 'necrotise fasciitis': 'SEVERE_INFECTION',
    'sepsis': 'SEVERE_INFECTION', 'meningococcal': 'SEVERE_INFECTION',
    'cellulitis': 'CELLULITIS_INFECTION', 'wound infection': 'CELLULITIS_INFECTION',
    'infected': 'CELLULITIS_INFECTION', 'tick bite': 'INFESTATION_BITE',
    'erythema migran': 'INFESTATION_BITE', 'shingle': 'INFESTATION_BITE',
    'eczema': 'DERMATITIS_RASH', 'rash': 'DERMATITIS_RASH', 'dermatitis': 'DERMATITIS_RASH',
    'sunburn': 'MINOR_CONDITIONS', 'ingrown': 'MINOR_CONDITIONS',

    # 2. CARDIO
    'nstemi': 'ACUTE_CORONARY', 'chest pain': 'ACUTE_CORONARY', 'exertional': 'ACUTE_CORONARY',
    'angina': 'ACUTE_CORONARY', 'dyspnoea': 'RESPIRATORY_CARDIAC', 'syncope': 'HEMODYNAMIC_INSTABILITY',
    'decompensated': 'HEMODYNAMIC_INSTABILITY', 'palpitations': 'HEMODYNAMIC_INSTABILITY',
    'atrial fibrillation': 'HEMODYNAMIC_INSTABILITY', 'blood pressure': 'HYPERTENSIVE_EVENT',
    'diaphoresis': 'HEMODYNAMIC_INSTABILITY',

    # 3. ENDO
    'hypoglycaemia': 'GLYCEMIC_CRISIS', 'hyperglycaemia': 'GLYCEMIC_CRISIS',
    'glucose': 'GLYCEMIC_CRISIS', 'diabetic': 'GLYCEMIC_CRISIS',
    'coma': 'ENDOCRINE_EMERGENCY', 'thyroid storm': 'ENDOCRINE_EMERGENCY',
    'adrenal': 'ENDOCRINE_EMERGENCY', 'cushing': 'METABOLIC_SYNDROME',
    'weight': 'METABOLIC_SYNDROME', 'thyroid': 'THYROID_DISORDER',

    # 4. ENT (Otorrino)
    'epiglottitis': 'AIRWAY_EMERGENCY', 'airway': 'AIRWAY_EMERGENCY', 'trismus': 'AIRWAY_EMERGENCY',
    'mastoiditis': 'ACUTE_INFECTION_ENT', 'tonsillitis': 'ACUTE_INFECTION_ENT',
    'sinusitis': 'ACUTE_INFECTION_ENT', 'epistaxis': 'NASAL_HEMORRHAGE',
    'foreign body': 'FOREIGN_BODY_ENT', 'ear pain': 'EAR_DISCOMFORT',
    'hearing': 'EAR_DISCOMFORT', 'runny nose': 'NASAL_SYMPTOMS',
    'nasal congestion': 'NASAL_SYMPTOMS', 'sore throat': 'THROAT_SYMPTOMS',

    # 5. GASTRO
    'haematemesis': 'GI_BLEED_UPPER', 'coffee-ground': 'GI_BLEED_UPPER', 'bleed': 'GI_BLEED_GENERAL',
    'acute abdomen': 'ACUTE_ABDOMEN_SURGICAL', 'obstruction': 'BOWEL_OBSTRUCTION',
    'hepatic failure': 'HEPATIC_CRISIS', 'pancreatitis': 'ACUTE_PANCREATITIS',
    'ischaemia': 'MESENTERIC_ISCHAEMIA', 'jaundice': 'BILIARY_LIVER_SYMPTOMS',
    'diarrhoe': 'DIARRHEA_GASTROENTERITIS', 'rigors': 'SYSTEMIC_INFECTION_GASTRO',
    'dyspepsia': 'FUNCTIONAL_GASTRO', 'constipation': 'FUNCTIONAL_GASTRO',
    'bloating': 'FUNCTIONAL_GASTRO', 'nausea': 'NAUSEA_VOMITING_GEN', 'vomiting': 'NAUSEA_VOMITING_GEN',

    # 6. GENITOURINARIO
    'urosepsis': 'UROSEPSIS_CRITICAL', 'ovarian torsion': 'GONADAL_TORSION_EMERGENCY',
    'testicular': 'GONADAL_TORSION_EMERGENCY', 'scrotal pain': 'GONADAL_TORSION_EMERGENCY',
    'pyelonephritis': 'ACUTE_PYELONEPHRITIS', 'renal colic': 'NEPHROLITHIASIS_COLIC',
    'haematuria': 'URINARY_BLEEDING', 'dysuria': 'UTI_SYMPTOMS',
    'urinary retention': 'ACUTE_URINARY_RETENTION', 'vaginal bleed': 'VAGINAL_BLEEDING',
    'genital discharge': 'GENITAL_INFECTION_STD', 'fever': 'FEVER_GU_ASSOCIATED',

    # 7. INFECTO
    'meningitis': 'CENTRAL_NERVOUS_SYSTEM_INF', 'mental status': 'CENTRAL_NERVOUS_SYSTEM_INF',
    'necrotising fasciitis': 'NECROTISING_SOFT_TISSUE_INF', 'dengue': 'TROPICAL_INF_DENGUE',
    'hepatitis': 'HEPATIC_INF', 'abscess': 'LOCALIZED_BACTERIAL_INF',
    'rigors': 'FEVER_SYNDROME', 'gastro': 'GASTROENTERITIS_INF',

    # 8. MSK (Musculoesquelético)
    'spinal cord': 'SPINAL_CORD_CRITICAL', 'vascular compromise': 'VASCULAR_EMERGENCY_MSK',
    'septic arthritis': 'SEPTIC_ART_EMERGENCY', 'fracture': 'BONE_FRACTURE',
    'distal radius': 'BONE_FRACTURE', 'pelvic': 'MAJOR_TRAUMA_PELVIC',
    'ankle': 'JOINT_INJURY_LOWER', 'strain': 'MUSCLE_STRAIN_SPRAIN', 'back pain': 'AXIAL_SKELETAL_PAIN',

    # 9. NEURO
    'status epilepticus': 'CRITICAL_SEIZURE_STATUS', 'thunderclap': 'ACUTE_CEREBROVASCULAR_HEM',
    'coma': 'CONSCIOUSNESS_LOSS_COMA', 'locked-in': 'BRAINSTEM_CRITICAL_SYNDROME',
    'facial droop': 'STROKE_SYMPTOMS_FOCAL', 'hand paraesthesia': 'STROKE_SYMPTOMS_FOCAL',
    'transient': 'TIA_TRANSIENT_ISCHAEMIA', 'seizure': 'EPILEPTIC_EVENT',
    'migraine': 'SEVERE_HEADACHE_MIGRAINE', 'vertigo': 'VESTIBULAR_VERTIGO', 'confusion': 'ALTERED_MENTAL_STATE',

    # 10. OFTALMO
    'sudden loss': 'SUDDEN_VISION_LOSS', 'central retinal': 'RETINAL_VASCULAR_EMERGENCY',
    'acute angle': 'ACUTE_GLAUCOMA_EMERGENCY', 'orbital cellulitis': 'ORBITAL_CELLULITIS_SEVERE',
    'foreign body': 'OCULAR_FOREIGN_BODY', 'corneal': 'CORNEAL_TRAUMA',
    'conjunctivitis': 'SUPERFICIAL_EYE_INF', 'photophobia': 'PHOTOPHOBIA_SYMPTOM',

    # 11. PSIQUIATRÍA
    'suicidal': 'SELF_HARM_SUICIDAL', 'self-harm': 'SELF_HARM_SUICIDAL', 'plan': 'SELF_HARM_SUICIDAL',
    'overdose': 'SUBSTANCE_EMERGENCY', 'intoxication': 'SUBSTANCE_EMERGENCY', 'withdrawal': 'SUBSTANCE_EMERGENCY',
    'psychosis': 'ACUTE_PSYCHOSIS', 'mania': 'ACUTE_PSYCHOSIS', 'panic attack': 'ANXIETY_PANIC',
    'insomnia': 'SLEEP_DISORDERS', 'depressive': 'DEPRESSIVE_SYMPTOMS',

    # 12. RESPIRATORIO
    'hypoxia': 'RESPIRATORY_FAILURE', 'tension': 'RESPIRATORY_FAILURE', 'acute severe': 'RESPIRATORY_FAILURE',
    'dyspnoea': 'ACUTE_DYSPNOEA', 'haemoptysis': 'AIRWAY_BLEEDING', 'pleuritic': 'CHEST_PAIN_RESP',
    'cough': 'CHRONIC_AIRWAY', 'night sweats': 'INFECTIOUS_RESP',

    # 13. TRAUMA
    'multi-system': 'CRITICAL_TRAUMA', 'unresponsive': 'CRITICAL_TRAUMA', 'stab': 'PENETRATING_WOUND',
    'head injury': 'HEAD_NEURO_TRAUMA', 'head bump': 'HEAD_NEURO_TRAUMA', 'rib fracture': 'BONE_FRACTURE_TRAUMA',
    'laceration': 'OPEN_WOUND_LACERATION', 'splinter': 'MINOR_FOREIGN_BODY', 'abrasion': 'SOFT_TISSUE_INJURY',

    # 14. OTROS
    'hypothermia': 'METABOLIC_CRITICAL', 'dehydration': 'METABOLIC_CRITICAL', 'electrolyte': 'METABOLIC_CRITICAL',
    'weight loss': 'SYSTEMIC_GENERAL_SYMPTOMS', 'fatigue': 'SYSTEMIC_GENERAL_SYMPTOMS',
    'substance': 'SUBSTANCE_RELATED_GENERAL', 'non-urgent': 'ADMINISTRATIVE_NON_CLINICAL',
    'fall': 'MECHANISM_OF_INJURY_FALL'
}

# ACTUALIZACIÓN
NUEVOS_MAPEOS = {
    # Síntomas Críticos que se escaparon
    'palpitation': 'HEMODYNAMIC_INSTABILITY',
    'rigor': 'FEVER_SYNDROME_SYSTEMIC',
    'asthma': 'CHRONIC_AIRWAY_DISEASE',
    'bite': 'INFESTATION_BITE',
    'insect': 'INFESTATION_BITE',
    'oedema': 'HEMODYNAMIC_LYMPHATIC',
    
    # Específicos de Especialidad
    'eye': 'SUPERFICIAL_EYE_INF',
    'conjunctival haemorrhage': 'RETINAL_VASCULAR_EMERGENCY',
    'foreign body sensation': 'OCULAR_FOREIGN_BODY',
    'skin rash': 'DERMATITIS_RASH',
    
    # Dolor y Trauma (estos son clave por su alto volumen)
    'abdominal pain': 'FUNCTIONAL_GASTRO', 
    'blunt abdominal trauma': 'CRITICAL_TRAUMA',
    'pleuritic chest pain': 'CHEST_PAIN_RESP',
    'local reaction': 'CELLULITIS_INFECTION'
}

MAPEO_MAESTRO_COMPLETO.update(NUEVOS_MAPEOS)

# ACTUALIZACIÓN: 
NUEVOS_RUIDOS = {
    'worsen', 'resolve', 'systemic feature', 'symptom', 'minor', 
    'suspect', 'vital', 'uncomplicate', 'symptomatic', 'complication',
    'localised', 'small', 'imbalance', 'systemic symptom', 'unclear', 
    'peripheral', 'feature'
}

RUIDO_MEDICO_TOTAL.update(NUEVOS_RUIDOS)

def clean_symptoms_final(lista_cruda):
    if not lista_cruda: return []
    categorias_detectadas = set()
    for s in lista_cruda:
        s_clean = s.lower().strip()
        if s_clean in RUIDO_MEDICO_TOTAL:
            continue
        for clave, categoria_maestra in MAPEO_MAESTRO_COMPLETO.items():
            if clave in s_clean:
                categorias_detectadas.add(categoria_maestra)
                break
    return list(categorias_detectadas)

# 3. EJECUCIÓN SOBRE EL DATAFRAME
d_texto['sintomas_limpios'] = d_texto['sintomas_lemmas'].apply(clean_symptoms_final)


In [ ]:
# 1. "Explotamos" la lista de síntomas para tener una fila por cada palabra extraída
todas_las_entidades = d_texto['sintomas_lemmas'].explode()

# 2. Obtenemos la lista de palabras únicas que nuestro MAPEO_MAESTRO sí reconoce
palabras_mapeadas = set(MAPEO_MAESTRO_COMPLETO.keys())

# 3. Filtramos: ¿Qué palabras extrajo SciSpacy que NO están en nuestro mapeo ni en el ruido?
# Esto nos dirá si hay síntomas importantes que ignoramos
sin_mapeo = todas_las_entidades[
    ~todas_las_entidades.str.lower().str.strip().isin(palabras_mapeadas) & 
    ~todas_las_entidades.str.lower().str.strip().isin(RUIDO_MEDICO_TOTAL)
]

# 4. Vemos el Top 30 de lo "ignorado"
print("Top 30 términos extraídos por SciSpacy que NO categorizamos:")
print(sin_mapeo.value_counts().head(30))



In [ ]:
d_texto.to_csv('triage_final_80k.csv', index=False)

In [ ]:
from IPython.display import FileLink

FileLink(r'triage_final_80k.csv')

In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        if 'triage' in filename:
            print(f"Ruta encontrada: {os.path.join(dirname, filename)}")

In [ ]:
############# LO VUELVO A LEER PARA CORRER CODIGO SOLO DE AQUI HACIA ABAJO UNA VEZ OBTENIDO TODOS LOS SINTOMAS DESDE EL TEXTO Y NO TENER QUE CORRER LO ANTERIOR. 
import pandas as pd

# Reemplaza lo que está entre comillas con el resultado de la celda de arriba
ruta_exacta = '/kaggle/input/datasets/camilacosentinoyanes/triage-final-80k/triage_final_80k.csv'

df_para_modelar = pd.read_csv(ruta_exacta)

print(f"¡Listo! Dataset cargado con {len(df_para_modelar)} filas.")
df_para_modelar.head()


In [ ]:
# ########## AHORA PASO ESAS VARIABLES DE SINTOMAS LIMPIOS FINALES A DUMMY (UN PACIENTE PUEDE TENER VARIOS SINTOMAS USAMOS ENTONCES MultiLabelBinarizer) Y UNO CON LOS DATOS DE ENTRADA DEL PACIENTE Y DE LA HISTORIA


# from sklearn.preprocessing import MultiLabelBinarizer

# # Inicializamos el binarizador para listas
# mlb = MultiLabelBinarizer()

# # Transformamos la columna de listas en un dataframe de 0 y 1
# # Esto creará columnas con los nombres reales: 'ACUTE_CORONARY', 'ANAPHYLAXIS_URTICARIA', etc.
# sintomas_dummies = pd.DataFrame(
#     mlb.fit_transform(df_para_modelar['sintomas_limpios']),
#     columns=mlb.classes_,
#     index=df_para_modelar.index
# )

# # Concatenamos con el resto de las variables (asegurándo de no incluir la original de listas)
# X_final = pd.concat([
#     df_para_modelar.drop(columns=['sintomas_limpios', 'sintomas_lemmas', 'triage_acuity', 'patient_id'], errors='ignore'),
#     sintomas_dummies
# ], axis=1)


# # Ahora sí, aplicamos get_dummies al RESTO de las variables categóricas (sexo, arrival_mode, etc.)
# X_final = pd.get_dummies(X_final, drop_first=True)

# # Limpiamos nombres de columnas para que XGBoost no proteste por caracteres especiales
# X_final.columns = [re.sub(r'[\[\]<>]', '_', str(col)) for col in X_final.columns]


# # Cargamos los otros datasets
# train = pd.read_csv('/kaggle/input/competitions/triagegeist/train.csv')
# hist = pd.read_csv('/kaggle/input/competitions/triagegeist/patient_history.csv')

# # Unimos 'train' con 'hist' para tener Info del Paciente + Antecedentes
# # Usamos 'left' para no perder registros del set de entrenamiento
# df_base = pd.merge(train, hist, on='patient_id', how='left')

# # Ahora unimos eso con tus síntomas procesados (df_para_modelar)
# df_consolidado = pd.merge(df_base, df_para_modelar[['patient_id', 'sintomas_limpios']], on='patient_id', how='left')

# # Finalmente, unimos con las dummies
# df_final = pd.concat([df_consolidado, sintomas_dummies], axis=1)

# print(f"Estructura final: {df_final.shape[0]} filas y {df_final.shape[1]} columnas.")



In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer
import re

# 1. Cargamos los datasets base (si no estaban cargados)
train = pd.read_csv('/kaggle/input/competitions/triagegeist/train.csv')
hist = pd.read_csv('/kaggle/input/competitions/triagegeist/patient_history.csv')

# 2. Unión de Información del Paciente + Antecedentes
df_base = pd.merge(train, hist, on='patient_id', how='left')

# 3. Preparación de Dummies de Síntomas (MultiLabelBinarizer)
# Importante: Usamos el ID como índice para que la unión sea perfecta
mlb = MultiLabelBinarizer()
sintomas_dummies = pd.DataFrame(
    mlb.fit_transform(df_para_modelar['sintomas_limpios']),
    columns=mlb.classes_,
    index=df_para_modelar['patient_id']  # Usamos patient_id de índice
)

# 4. CONSOLIDACIÓN FINAL (Evitando la columna de listas)
# Seteamos index en df_base para unir por ID
df_final = df_base.set_index('patient_id').join(sintomas_dummies, how='left')

# 5. Get Dummies para el resto de variables categóricas (sexo, modo de llegada, etc.)
# Esto solo afectará a las columnas tipo 'object' o 'category'
df_final = pd.get_dummies(df_final, drop_first=True)

# 6. LIMPIEZA DE COLUMNAS (Elimina caracteres especiales y basura de 1 sola letra)
# Primero: Limpiar caracteres que XGBoost no acepta
df_final.columns = [re.sub(r'[\[\]<>]', '_', str(col)) for col in df_final.columns]

# Segundo: El filtro "anti-letras-sueltas" (Solo dejamos columnas con más de 1 caracter)
# Esto vuela las 'A', 'B', '[', ']', ',', etc.
df_final = df_final.loc[:, [col for col in df_final.columns if len(str(col)) > 1]]

# 7. Reset del índice para recuperar 'patient_id' como columna si es necesario
df_final = df_final.reset_index()

# 8. Limpieza final de filas sin target (por si hubo algún descalce en el merge)
df_final = df_final.dropna(subset=['triage_acuity'])

print(f"¡Logrado! Estructura final: {df_final.shape[0]} filas y {df_final.shape[1]} columnas.")
# print("Primeras 5 columnas para verificar:", df_final.columns[:5].tolist())

In [ ]:
df_final = df_final.dropna(subset=['triage_acuity'])

In [ ]:
df_final.shape[0]

In [ ]:
df_final.columns

In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier

# 1. Preparación de X e y
# Quitamos lo que no es predictor. 'errors=ignore' por si alguna ya se borró antes.
X = df_final.drop(columns=['sintomas_limpios', 'triage_acuity', 'patient_id', 'disposition'], errors='ignore')
y = df_final['triage_acuity']

# Lista negra de variables del futuro o ruidosas
columnas_a_quitar = [
    'triage_acuity',    # Es el target
    'patient_id',       # Es un identificador
    'ed_los_hours',     # LEAKAGE: Se sabe al final
    'disposition',      # LEAKAGE: Se sabe al final
    'triage_nurse_id',  # SESGO: Evita aprender mañas de enfermeros específicos
    'site_id',           # OPCIONAL: Sacalo si querés que el modelo funcione en cualquier hospital
    'disposition_deceased', 'disposition_discharged', 'disposition_lama',
       'disposition_lwbs', 'disposition_observation',
       'disposition_transferred'
]

# Aplicamos el drop (usamos errors='ignore' por si alguna ya la borraste antes)
X = df_final.drop(columns=columnas_a_quitar, errors='ignore')


# 2. Transformación a Dummies
# Convertimos variables categóricas (como arrival_mode o sex) a columnas 0/1.
# El resto (numéricas como news2_score) quedan igual.
X_final = pd.get_dummies(X, drop_first=True)

# 3. Limpieza de nombres de columnas
# XGBoost es muy especial con los caracteres: prohibidos [, ], <, >
def clean_column_names(df):
    new_cols = [re.sub(r'[\[\]<>]', '_', str(col)) for col in df.columns]
    df.columns = new_cols
    return df

X_final = clean_column_names(X_final)

# 4. Aseguramos que TODO sea numérico
# Si quedó algún residuo de texto (como fechas o IDs raros), esto lo fuerza a número o lo hace 0.
X_final = X_final.apply(pd.to_numeric, errors='coerce').fillna(0)

# 5. Encoding del Target
# Triage_acuity viene como 1, 2, 3, 4, 5. XGBoost necesita 0, 1, 2, 3, 4.
le = LabelEncoder()
y_numpy = le.fit_transform(y)

# 6. Conversión a Matrices NumPy
# Esto es más eficiente para XGBoost y evita errores de índices de Pandas.
X_numpy = X_final.values

# 7. Split Estratificado
# Importante: 'stratify=y_numpy' asegura que la proporción de urgencias sea igual en train y test.
X_train, X_test, y_train, y_test = train_test_split(
    X_numpy, y_numpy, 
    test_size=0.2, 
    random_state=42, 
    stratify=y_numpy
)

# 8. Entrenamiento del Modelo (Hiperparámetros Optimizados)
modelo = XGBClassifier(
    n_estimators=500,        # Más árboles para mayor capacidad de aprendizaje
    learning_rate=0.01,      # Pasos más finos para no "pasarse" del óptimo
    max_depth=5,             # Profundidad para capturar interacciones entre síntomas
    subsample=0.8,           # Regularización: usa 80% de filas por árbol
    colsample_bytree=0.8,    # Regularización: usa 80% de variables por árbol
    objective='multi:softprob',
    random_state=42,
    tree_method='hist',      # Método rápido para grandes volúmenes de datos
    n_jobs=-1                # Usa toda la potencia de la CPU disponible
)

print("Iniciando entrenamiento sobre los 80,000 registros...")
modelo.fit(X_train, y_train)

print(f"¡AL FIN! Modelo entrenado con éxito.")
print(f"Variables procesadas: {X_final.shape[1]}")

In [ ]:
########## HAGO VALIDACION CRUZADA 
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer, f1_score, recall_score, roc_auc_score
from xgboost import XGBClassifier
import numpy as np

# Definimos el modelo
# Usamos un modelo base. 
modelo_cv = XGBClassifier(
    n_estimators=500, 
    max_depth=5, 
    learning_rate=0.01, 
    random_state=42,
    use_label_encoder=False,
    eval_metric='mlogloss'
)

# Configuración de Stratified K-Fold
# 5 folds es un estándar balanceado entre tiempo y robustez para 80k datos
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Definimos las métricas que queremos monitorear
# El 'f1_macro' es excelente para ver cómo rinde el modelo en todas las categorías de urgencia
scoring = {
    'f1_macro': 'f1_macro',
    'accuracy': 'accuracy',
    'recall_macro': 'recall_macro'
}

print("Iniciando Validación Cruzada... Esto puede demorar unos minutos.")

# Ejecución de la Validación Cruzada
cv_results = cross_validate(
    modelo_cv, X_train, y_train, 
    cv=skf, 
    scoring=scoring, 
    return_train_score=False,
    n_jobs=-1  # Usamos todos los núcleos disponibles para ir más rápido
)

# 6. Reporte de Resultados
print("-" * 30)
print("RESULTADOS DE VALIDACIÓN CRUZADA (Promedios):")
print(f"F1-Score (Macro): {np.mean(cv_results['test_f1_macro']):.4f} (+/- {np.std(cv_results['test_f1_macro']):.4f})")
print(f"Recall (Macro):   {np.mean(cv_results['test_recall_macro']):.4f}")
print(f"Accuracy:         {np.mean(cv_results['test_accuracy']):.4f}")
print("-" * 30)

In [ ]:
################ PREPARAMOS PARA KAGGLE
# Cargar el set de test de la competencia
test_comp = pd.read_csv('/kaggle/input/competitions/triagegeist/test.csv')

# Unir con los mismos datos que usamos en train (historia y tus síntomas de test)
# Asumiendo que procesaste los síntomas de test en 'df_test_procesado'
df_submission_base = pd.merge(test_comp, hist, on='patient_id', how='left')
df_submission_base = pd.merge(df_submission_base, df_para_modelar, on='patient_id', how='left')

# Aplicar Get Dummies para tener las mismas columnas que X_train
X_test_final = pd.get_dummies(df_submission_base.drop(columns=['patient_id', 'sintomas_limpios'], errors='ignore'))

# ALINEAR COLUMNAS: Este paso es VITAL
# Asegura que el set de test tenga las mismas columnas que el de train, en el mismo orden
# Si falta alguna categoría en test, le pone 0.
X_test_final = X_test_final.reindex(columns=X_final.columns, fill_value=0)

# Limpieza de nombres y conversión a NumPy (igual que hicimos para el fit)
X_test_final = clean_column_names(X_test_final)
X_test_final = X_test_final.apply(pd.to_numeric, errors='coerce').fillna(0)
X_test_numpy = X_test_final.values

# Predecir
preds_encoded = modelo.predict(X_test_numpy)

# Volver al formato 1-5 (deshacer el LabelEncoder)
preds_originales = le.inverse_transform(preds_encoded)

# Crear el archivo final
submission = pd.DataFrame({
    'patient_id': test_comp['patient_id'],
    'triage_acuity': preds_originales
})

#submission.to_csv('submission.csv', index=False)
print("¡Archivo submission.csv listo para descargar!")



In [ ]:
print(df_submission_base['sintomas_limpios'].isnull().sum())

In [ ]:
submission.to_csv('submission.csv', index=False)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Obtenemos las importancias (los valores numéricos)
importancias_valores = modelo.feature_importances_

# 2. Creamos un DataFrame usando los nombres de las columnas de tu matriz de entrenamiento
# IMPORTANTE: X_final debe ser la que usaste justo antes del split
df_imp = pd.DataFrame({
    'Variable': X_final.columns,
    'Importancia': importancias_valores
})

# 4. Graficamos el Top 15
plt.figure(figsize=(10, 8))
top_15 = df_imp.sort_values(by='Importancia', ascending=False).head(15)

sns.barplot(data=top_15, x='Importancia', y='Variable', palette='viridis')
plt.title('Factores Determinantes en la Prioridad de Triage', fontsize=14, fontweight='bold')
plt.xlabel('Importancia Relativa', fontsize=12)
plt.ylabel('Indicador Clínico / Síntoma', fontsize=12)
plt.tight_layout()

plt.show()

In [ ]:
df_final.columns.to_series().to_csv('nomVariables.csv', index=False)